# 2 — Write: dispatch the fleet with `zagg.client`, stage-timed

[![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/englacial/zagg/main?urlpath=lab/tree/notebooks/02_dispatch_fleet.ipynb)

_Runs end-to-end on [Binder](https://mybinder.org/v2/gh/englacial/zagg/main?urlpath=lab/tree/notebooks/02_dispatch_fleet.ipynb)
**in its default stub mode**: the fan-out is driven by an injected stub Lambda
client defined in this notebook, so the whole `zagg.client` API — futures,
progress bar, error surfacing, per-phase worker splits, the post-run tail —
runs with no AWS account, no credentials, and no cost. Set `USE_REAL_FLEET =
True` (section 0) to point the same cells at the deployed fleet._

The second of three narrative notebooks
([#328](https://github.com/englacial/zagg/issues/328)). Notebook 1 built the
shard map; this one fans it out, one Lambda worker per shard, and times:

- **dispatch wall** — how long `dispatch()` takes to return (the setup
  handshake, then the fan-out goes to the background),
- **fleet completion** — wall from dispatch to the last shard resolving,
- **per-phase worker splits** — the `read` / `index` / `aggregate` / `write`
  breakdown the workers report back, summed across the fleet.

## Why a stub, and what it does and does not prove

Lambda invocation cannot be anonymous, so a notebook that only ever talked to
the real fleet could not run on Binder at all (CLAUDE.md §4). `Run.from_config`
takes a `lambda_client=` seam — the same injection point
`tests/test_client.py` uses — so the notebook ships **dual-mode**: the default
path answers every invoke from an in-notebook stub shaped like the deployed
worker's response envelope.

What the stub mode genuinely exercises: the real `Run` / `RunHandle` /
`ShardError` code paths, the real event construction, the real shard ordering
and thread pool, the real post-run tail invokes, the real progress and status
surface. What it cannot: the worker itself, IAM, S3, and the actual numbers.
**Treat stub timings as a shape, not a measurement** — the fleet numbers are
what section 7 records, and only `USE_REAL_FLEET = True` produces them.

## The API

```python
from zagg.client import Run

run = Run.from_config(config, shardmap=..., store="s3://...")
handle = run.dispatch()                 # returns after the setup handshake
for fut in handle.progress():           # tqdm.auto over as_completed
    r = fut.result()                    # worker envelope, or raises ShardError
handle.status()                         # ('pending', 'ok', 'failed')
```

This is the facade ratified on
[#265](https://github.com/englacial/zagg/issues/265#issuecomment-5124017044)
and shipped by [#326](https://github.com/englacial/zagg/issues/326) — the same
composition `.github/scripts/run_benchmark.py` runs, so the notebook cannot
drift from what CI tests.

## 0. Mode switch

`USE_REAL_FLEET = False` is the Binder/default path — stub client, no
credentials, no AWS calls, nothing written anywhere.

`USE_REAL_FLEET = True` dispatches for real. It needs, all of them:

1. AWS credentials with `lambda:InvokeFunction` on the deployed worker,
2. `ZAGG_LAMBDA_FUNCTION_NAME` set to that function (or a full cross-account
   ARN — see `cryocloud_example.ipynb`),
3. a writable `STORE` you own, and
4. NASA Earthdata credentials for the *workers'* source reads (leave
   `SOURCE_CREDENTIALS = None` and the config's credentials provider resolves
   them, exactly as `zagg.runner.agg` does).

It **costs money** — the ceiling is printed in section 3 before anything is
invoked.

In [ ]:
from pathlib import Path

from zagg.notebook import StageTimer

USE_REAL_FLEET = False  # <- the one switch; see section 0 before flipping it


def repo_file(rel):
    """Resolve a repo file whether we run from the repo root or notebooks/."""
    for base in (Path.cwd(), Path.cwd().parent):
        if (base / rel).exists():
            return str(base / rel)
    raise FileNotFoundError(rel)


# The live per-merge benchmark target: ATL03 photon-height t-digests on a
# HEALPix o9/o19 hive store. Same config and shard map notebook 1 rebuilds.
CONFIG = repo_file("tests/data/benchmark/configs/atl03_tdigest_healpix_o9_hive.yaml")
SHARDMAP = repo_file("tests/data/benchmark/shardmaps/sm_healpix_o9.json")

if USE_REAL_FLEET:
    STORE = "s3://your-bucket/zagg-notebook/tdigest_healpix_o9_hive.zarr"
    FUNCTION_NAME = None  # None -> ZAGG_LAMBDA_FUNCTION_NAME + the config's worker variant
    SOURCE_CREDENTIALS = None  # None -> resolve the config's provider (NSIDC)
else:
    # Nothing is ever written here: every store write in zagg rides a worker
    # invoke (the dispatcher-never-writes invariant), and the stub answers
    # every invoke in-process. The path only has to look like an s3 store.
    STORE = "s3://zagg-stub-demo/tdigest_healpix_o9_hive.zarr"
    FUNCTION_NAME = "process-shard-stub"
    SOURCE_CREDENTIALS = {
        "accessKeyId": "STUB",
        "secretAccessKey": "STUB",
        "sessionToken": "STUB",
    }

timer = StageTimer("write (fleet)" if USE_REAL_FLEET else "write (stub)")
print(f"mode:     {'REAL FLEET' if USE_REAL_FLEET else 'stub (anonymous)'}")
print(f"config:   {CONFIG}")
print(f"shardmap: {SHARDMAP}")
print(f"store:    {STORE}")

## 1. The stub worker

Shaped like boto3's Lambda client where `zagg` touches it — one `invoke(**kw)`
returning `{"Payload": <reader>, "FunctionError": ...}` — and like the deployed
worker where `zagg` reads it: a `{"statusCode", "body"}` envelope whose body
carries `total_obs`, `duration_s`, and the `phase_timings` split.

It also answers the **`mode`** invokes — `ping`, `setup`, `finalize`,
`coverage`, `stats`, `sweep` — which are the store writes the dispatcher never
does itself. Section 6 prints which of those actually fired.

Two shards are rigged to show the error surface: one hard failure (surfaces as
`ShardError`) and one *benign* no-work outcome (`"No granules found"`), which
resolves **normally** and counts as `ok` — matching how the `agg` path counts
it. Per-shard sleep scales with the shard's granule count, so the progress bar
completes out of order the way a real fan-out does.

In [ ]:
import json
import threading
import time


class _Payload:
    """boto3 returns a streaming body; the reader only calls .read()."""

    def __init__(self, raw):
        self._raw = raw

    def read(self):
        return self._raw


class StubLambdaClient:
    """A boto3-Lambda-shaped stand-in for the deployed zagg worker."""

    def __init__(self, *, fail=(), benign=(), seconds_per_granule=0.04):
        self.events = []
        self._lock = threading.Lock()
        self._fail = set(fail)
        self._benign = set(benign)
        self._per_granule = seconds_per_granule

    def _envelope(self, body, status=200):
        raw = json.dumps({"statusCode": status, "body": json.dumps(body)}).encode()
        return {"Payload": _Payload(raw), "FunctionError": None}

    def invoke(self, **kwargs):
        event = json.loads(kwargs["Payload"])
        with self._lock:
            self.events.append((kwargs["InvocationType"], event))

        if event.get("mode") is not None:
            # ping / setup / finalize / coverage / stats / sweep — the
            # worker-side store writes. Nothing to fake beyond a 200.
            return self._envelope({"zagg_version": "stub"})

        shard_key = int(event["shard_key"])
        n_granules = len(event.get("granule_urls") or [])
        work_s = self._per_granule * max(n_granules, 1)
        time.sleep(work_s)

        if shard_key in self._fail:
            return self._envelope({"error": "Runtime.OutOfMemory (stub)"}, status=500)
        if shard_key in self._benign:
            # A no-work outcome: resolves normally, counted as ok, not an error.
            return self._envelope({"error": "No granules found"})
        return self._envelope(
            {
                "shard_key": shard_key,
                "total_obs": 250_000 * max(n_granules, 1),
                "cells_with_data": 40_000,
                "duration_s": work_s,
                # The split a profile=True worker reports (worker.py).
                "phase_timings": {
                    "read": work_s * 0.62,
                    "index": work_s * 0.11,
                    "aggregate": work_s * 0.19,
                    "write": work_s * 0.08,
                },
            }
        )

    def modes(self):
        return [e["mode"] for _t, e in self.events if e.get("mode") is not None]

In [ ]:
shard_keys = json.loads(Path(SHARDMAP).read_text())["shard_keys"]
print(f"{len(shard_keys)} shards in the map")

if USE_REAL_FLEET:
    lambda_client = None  # boto3 builds one, sized to the fan-out
else:
    lambda_client = StubLambdaClient(fail=[shard_keys[2]], benign=[shard_keys[3]])
    print(f"stub: shard {shard_keys[2]} fails, shard {shard_keys[3]} returns no work")

## 2. Build the run

`Run.from_config` does the composition — load/validate the config, load the
shard map, check its grid signature against the config's grid, resolve the
store root and the worker function name — and refuses anything outside the v1
scope (temporal, raster, and windowed configs go through `zagg.runner.agg`).

Nothing is invoked yet.

In [ ]:
from zagg.client import Run
from zagg.config import load_config

config = load_config(CONFIG)

run = Run.from_config(
    config,
    shardmap=SHARDMAP,
    store=STORE,
    function_name=FUNCTION_NAME,
    lambda_client=lambda_client,
    source_credentials=SOURCE_CREDENTIALS,
    profile=True,  # have the workers report the read/index/aggregate/write split
)
run

## 3. The cost ceiling, before any invoke

`max_cost_preview` resolves the pre-invoke ceiling from the shard map + config
alone — the same unit accounting the dispatcher uses, no AWS access. The
notebook path **displays** it and never blocks on it (ratified on
[#298](https://github.com/englacial/zagg/issues/298)); the blocking yes/no gate
is CLI-only (`python -m zagg ... --yes`).

It is a *ceiling*: every worker billed for the full timeout. Real runs come in
far under it — see `cost_reporting.ipynb` for the max → estimated → actual
progression.

In [ ]:
from zagg.notebook import format_max_cost, max_cost_preview

preview = max_cost_preview(config, SHARDMAP)
print(format_max_cost(preview))
preview

## 4. Stage — dispatch wall

`dispatch()` returns as soon as the **setup handshake** completes: on a hive
store that is a fail-fast ping plus the fire-and-forget manifest write, both
worker invokes (the dispatcher writes nothing itself). The fan-out then runs in
the background and each shard's outcome arrives through its future.

So this stage times the handshake, not the work — a couple of hundred
milliseconds against a real fleet. Everything after it is section 5.

`max_workers` is the fan-out width. Left unset against a real fleet it is sized
by an account-concurrency preflight; it is pinned here so the stub run is
deterministic and the cell reads the same in both modes.

In [ ]:
if not USE_REAL_FLEET:
    # Convenience only -- the stub run is CORRECT without this. The post-run
    # tail verifies the run-stats parquet landed by polling the store read-only
    # (a 20 s window, then one re-fire); that poll is fail-open, so against a
    # store that does not exist it simply reports "absent" and the run finishes
    # either way. Zeroing the window just skips ~40 s of pointless polling on
    # the finisher thread that the harvest loop below joins.
    #
    # It is a private module global, which is not a pattern to copy -- whether
    # zagg.client should expose a public knob is an open question on PR #350.
    from zagg import runner

    runner._RUN_STATS_VERIFY_WINDOW_S = 0

with timer.stage("dispatch"):
    handle = run.dispatch(max_workers=8)

print(repr(handle))
print(f"store: {handle.store_path}")

## 5. Stage — fleet completion

`handle.progress()` is `as_completed` under a `tqdm.auto` bar, one tick per
shard (tqdm rides the `analysis` extra, [ratified on
#265](https://github.com/englacial/zagg/issues/265#issuecomment-5124017044); it
is never in the `lambda` extra — workers must not import it). `handle.as_completed()`
is the bar-free equivalent.

Each future resolves to the worker's result dict, or raises `ShardError`
carrying that shard's full payload — timings and the worker's error body
survive into the exception, so a failed shard is diagnosable without going to
CloudWatch.

**Draining this loop is what finishes the run**: exhausting a harvest iterator
joins the post-run tail (the finalize backstop plus the coverage / run-stats /
sweep rollups, all worker invokes) and re-raises a finalize failure. So the
loop below *is* the whole run — `handle.wait()` is only needed for explicit
control, like a timeout.

In [ ]:
from zagg.client import ShardError

results, failures = {}, {}

with timer.stage("fleet completion"):
    for fut in handle.progress():
        try:
            r = fut.result()
            results[r["shard_key"]] = r
        except ShardError as e:
            failures[e.shard_key] = e

print(f"\nstatus: {handle.status()}")
print(f"{len(results)} resolved, {len(failures)} failed")

In [ ]:
# Per-shard outcome. `wall_time` is the client-side round trip; `duration_s`
# (in the body) is what the worker itself reports.
for key, r in sorted(results.items(), key=lambda kv: -kv[1]["wall_time"]):
    body = r["body"]
    note = body.get("error") or f"{body.get('total_obs', 0):,} obs"
    print(
        f"  shard {key:>20d}  {r['granule_count']:>3d}g  "
        f"wall {r['wall_time']:6.2f}s  retries {r['retries']}  {note}"
    )

for key, e in failures.items():
    print(f"  shard {key:>20d}  FAILED: {e.payload.get('error')}")

## 6. Per-phase worker splits

With `profile=True` each worker reports where its time went:

| phase | what it is |
| --- | --- |
| `read` | byte-range HDF5 reads of the shard's granules |
| `index` | morton/HEALPix assignment of the observations |
| `aggregate` | the reducers (here: building a t-digest per cell) |
| `write` | the leaf zarr write |

Summing across the fleet is the split that says whether a run is I/O-bound or
compute-bound — the same numbers the benchmark series records per merge.

In [ ]:
phases = {}
for r in results.values():
    for name, secs in (r["body"].get("phase_timings") or {}).items():
        phases[name] = phases.get(name, 0.0) + float(secs)

worker_total = sum(phases.values())
if worker_total:
    for name, secs in sorted(phases.items(), key=lambda kv: -kv[1]):
        print(f"  {name:<10} {secs:8.2f}s  {100 * secs / worker_total:5.1f}%")
    print(f"  {'total':<10} {worker_total:8.2f}s   (summed across workers)")
else:
    print("no phase timings reported (the run was not dispatched with profile=True)")

In [ ]:
# The worker-invoke tail: every store write in a zagg run rides one of these.
if not USE_REAL_FLEET:
    print("mode invokes the stub answered:", lambda_client.modes())

## 7. Stage timings

Dispatch is the handshake; fleet completion is the fan-out. Against a real
fleet the ratio is the interesting part — a dispatch wall that grows with shard
count means the handshake is being serialized, and a fleet wall far above the
slowest single worker means the fan-out is concurrency-starved rather than
worker-slow.

In [ ]:
print(timer.summary())

In [ ]:
timer.as_dict()

## 8. The v2 event transport (real fleet only)

`dispatch(transport="event")` swaps the transport underneath the same API
([#327](https://github.com/englacial/zagg/issues/327), merged in PR #343):
fire-and-forget `InvocationType="Event"` invokes resolved by a single poller
thread reading the run's status-object prefix. No held connections, a retry
policy split by fault class, and — the reason it matters for a notebook — the
run **survives the client**: a kernel restart can reattach by run id.

The stub cannot serve this path (it resolves futures from status objects in a
real store, not from invoke responses), so this cell is inert in stub mode.

In [ ]:
if USE_REAL_FLEET:
    handle2 = run.dispatch(transport="event", max_workers=64)
    for fut in handle2.progress():
        fut.result()

    # ... and from a fresh kernel, adopt the same fleet by run id:
    #   from zagg.client import Run
    #   handle2 = Run.attach(STORE, run_id)
else:
    print('event transport needs a deployed worker + a real store; set USE_REAL_FLEET = True')

## Next

**[`03_read_tensors.ipynb`](03_read_tensors.ipynb)** — read a t-digest product
back and decode it to `(tensor, mask, (offset, gain), morton_id)` blocks, with
fetch / decode / vertical-rasterize timed separately, and a percentile-surface
(bare-earth / canopy) render on top.